# 03_lstm_baseline

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.append("/content/drive/MyDrive")

In [3]:
from datasets import load_from_disk
dataset = load_from_disk("/content/drive/MyDrive/data/processed/iitb_filtered")

In [4]:
SUBSET_SIZE = 180_000  # adjust as needed

train_subset = dataset["train"].shuffle(seed=42).select(range(SUBSET_SIZE))

In [5]:
val_subset = dataset["validation"].shuffle(seed=42).select(range(min(10_000, len(dataset["validation"]))))
test_subset = dataset["test"].shuffle(seed=42).select(range(min(10_000, len(dataset["test"]))))

In [6]:
train_en = [ex["en"] for ex in train_subset]
train_hi = [ex["hi"] for ex in train_subset]

In [7]:
val_en = [ex["en"] for ex in val_subset]
val_hi = [ex["hi"] for ex in val_subset]

In [8]:
test_en = [ex["en"] for ex in test_subset]
test_hi = [ex["hi"] for ex in test_subset]

In [9]:
# import sys
# sys.path.append("src")

In [10]:
sys.path.append("/content/drive/MyDrive/src")

In [11]:
import pickle
from src.vocab import Vocab

with open("/content/drive/MyDrive/data/processed/en_vocab.pkl", "rb") as f:
    en_vocab = pickle.load(f)

with open("/content/drive/MyDrive/data/processed/hi_vocab.pkl", "rb") as f:
    hi_vocab = pickle.load(f)

print(type(en_vocab), len(en_vocab))
print(type(hi_vocab), len(hi_vocab))

<class 'vocab.Vocab'> 114395
<class 'vocab.Vocab'> 105200


In [12]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [13]:
from dataset import TranslationDataset, collate_fn
from torch.utils.data import DataLoader
from functools import partial

train_dataset = TranslationDataset(train_en, train_hi, en_vocab, hi_vocab)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                     collate_fn=partial(collate_fn, pad_idx=en_vocab.PAD_IDX), num_workers=2, pin_memory=True)

src, tgt, src_lens, tgt_lens = next(iter(train_loader))
print(src.shape, tgt.shape)

torch.Size([32, 53]) torch.Size([32, 49])


In [14]:
val_dataset = TranslationDataset(val_en, val_hi, en_vocab, hi_vocab)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,
                        collate_fn=partial(collate_fn, pad_idx=en_vocab.PAD_IDX), num_workers=2, pin_memory=True)

src, tgt, src_lens, tgt_lens = next(iter(val_loader))
print(src.shape, tgt.shape)

torch.Size([32, 43]) torch.Size([32, 47])


In [15]:
from models.lstm_se12seq import Encoder, Decoder, Seq2Seq

EMBED_DIM = 128
HIDDEN_DIM = 256

encoder = Encoder(len(en_vocab), EMBED_DIM, HIDDEN_DIM, pad_idx=en_vocab.PAD_IDX).to(device)
decoder = Decoder(len(hi_vocab), EMBED_DIM, HIDDEN_DIM, pad_idx=hi_vocab.PAD_IDX).to(device)
model = Seq2Seq(encoder, decoder, device, start_idx=hi_vocab.START_IDX, end_idx=hi_vocab.END_IDX).to(device)

In [16]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=hi_vocab.PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [17]:
from train import train_epoch, evaluate_epoch

NUM_EPOCHS = 1

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, teacher_forcing_ratio=0.5)
    val_loss = evaluate_epoch(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} — Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}")

Epoch 1/1 — Train loss: 6.7697, Val loss: 7.1730


In [26]:
for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, teacher_forcing_ratio=0.5)
    val_loss = evaluate_epoch(model, val_loader, criterion, device)
    print(f"Epoch {epoch+2}/{NUM_EPOCHS+1} — Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}")

Epoch 2/2 — Train loss: 6.0446, Val loss: 7.0782


In [27]:
from train import translate_sentence

predictions = []
references = []

for ex in test_subset:
    pred = translate_sentence(model, ex["en"], en_vocab, hi_vocab, device)
    predictions.append(pred)
    references.append(ex["hi"])

In [28]:
import sacrebleu
bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"BLEU score: {bleu.score:.2f}")

BLEU score: 0.09


In [30]:
import json

lstm_results = {
    "model": "LSTM Seq2Seq (no attention)",
    "epochs_trained": 2,
    "embed_dim": 128,
    "hidden_dim": 256,
    "subset_size": 180_000,
    "bleu_score": bleu.score,
}

with open("lstm_baseline_results.json", "w") as f:
    json.dump(lstm_results, f, indent=2, ensure_ascii=False)

In [31]:
torch.save(model.state_dict(), "model.pt")